In [2]:
# ── dependencies ────────────────────────────────────────────────────────────
# pip install rank_bm25 scikit-learn tqdm
import json, random, math
from pathlib import Path
from collections import defaultdict

import numpy as np
from tqdm import tqdm
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from rank_bm25 import BM25Okapi

ModuleNotFoundError: No module named 'numpy'

In [ ]:
DATA_DIR       = Path("data/processed")
PASSAGES_FILE  = DATA_DIR / "msmarco_small_passages.json"
QUERIES_FILE   = DATA_DIR / "msmarco_small_queries.json"
QRELS_FILE     = DATA_DIR / "msmarco_small_qrels.json"
TEST_PAIRS     = DATA_DIR / "test_pairs.jsonl"

def load_json(path):
    with open(path, encoding="utf-8") as f:
        return json.load(f)

def load_jsonl(path):
    with open(path, encoding="utf-8") as f:
        return [json.loads(l) for l in f]

passages_raw = load_json(PASSAGES_FILE)   # list of {id, text}
queries_raw  = load_json(QUERIES_FILE)    # list of {id, text}
qrels        = load_json(QRELS_FILE)      # {qid: [[pid, score], ...]}
test_pairs   = load_jsonl(TEST_PAIRS)     # [{query, positive}]

# Build lookup dicts
pid2text = {p["id"]: p["text"] for p in passages_raw}
pid2idx  = {p["id"]: i        for i, p in enumerate(passages_raw)}
idx2pid  = {i: p["id"]        for i, p in enumerate(passages_raw)}
corpus   = [p["text"] for p in passages_raw]   # ordered list of passage texts

print(f"Passages : {len(corpus):,}")
print(f"Queries  : {len(queries_raw):,}")
print(f"Test pairs: {len(test_pairs):,}")

In [ ]:
def mrr_at_k(ranked_pids: list, relevant_pids: set, k: int) -> float:
    """Mean Reciprocal Rank at k for a single query."""
    for rank, pid in enumerate(ranked_pids[:k], start=1):
        if pid in relevant_pids:
            return 1.0 / rank
    return 0.0

def recall_at_k(ranked_pids: list, relevant_pids: set, k: int) -> float:
    """Recall at k for a single query."""
    hits = sum(1 for pid in ranked_pids[:k] if pid in relevant_pids)
    return hits / len(relevant_pids) if relevant_pids else 0.0

def evaluate(retrieve_fn, queries, qrels, ks=(10, 50)):
    """
    retrieve_fn(query_text) -> [pid, pid, ...]  ordered best-first
    Returns dict of metric -> score.
    """
    metrics = defaultdict(list)
    for q in tqdm(queries, desc="Evaluating"):
        qid   = q["id"]
        qtext = q["text"]
        if qid not in qrels:
            continue
        relevant = {pid for pid, _ in qrels[qid]}
        ranked   = retrieve_fn(qtext)
        for k in ks:
            metrics[f"MRR@{k}"].append(mrr_at_k(ranked, relevant, k))
            metrics[f"Recall@{k}"].append(recall_at_k(ranked, relevant, k))
    return {m: float(np.mean(v)) for m, v in metrics.items()}

In [ ]:
print("Building TF-IDF index …")
tfidf = TfidfVectorizer(
    lowercase=True,
    stop_words="english",
    ngram_range=(1, 2),    # unigrams + bigrams
    max_features=200_000,
    sublinear_tf=True,     # log(1+tf) — reduces impact of very frequent terms
)
passage_matrix = tfidf.fit_transform(corpus)  # shape: (n_passages, vocab)
print(f"Matrix shape: {passage_matrix.shape}  (sparse)")

def tfidf_retrieve(query_text: str, top_k: int = 100) -> list:
    q_vec  = tfidf.transform([query_text])         # (1, vocab)
    scores = cosine_similarity(q_vec, passage_matrix).flatten()  # (n_passages,)
    top_idxs = np.argsort(scores)[::-1][:top_k]
    return [idx2pid[i] for i in top_idxs]

# Quick sanity check
sample_query = test_pairs[0]["query"]
sample_pos   = test_pairs[0]["positive"]
results      = tfidf_retrieve(sample_query, top_k=5)
print(f"\nQuery: {sample_query}")
print(f"Top-5 PIDs: {results}")
print(f"Top-1 text: {pid2text.get(results[0], 'N/A')[:200]}")

In [ ]:
# Build a small query list from test_pairs for quick eval
# (Use full queries_raw for full eval — may be slow on large corpora)
test_query_ids = set()

# Match test pairs back to query objects
text2qid = {q["text"]: q["id"] for q in queries_raw}
test_queries = []
for pair in test_pairs:
    qid = text2qid.get(pair["query"])
    if qid and qid in qrels:
        test_queries.append({"id": qid, "text": pair["query"]})
test_queries = list({q["id"]: q for q in test_queries}.values())  # deduplicate
print(f"Test queries for eval: {len(test_queries):,}")

tfidf_scores = evaluate(tfidf_retrieve, test_queries, qrels)
print("\n── TF-IDF Results ──")
for metric, val in sorted(tfidf_scores.items()):
    print(f"  {metric}: {val:.4f}")

In [ ]:
import re

def simple_tokenize(text: str) -> list:
    """Lowercase + split on non-alphanumeric."""
    return re.findall(r"[a-z0-9]+", text.lower())

print("Tokenising corpus for BM25 …")
tokenised_corpus = [simple_tokenize(doc) for doc in tqdm(corpus)]

print("Building BM25 index …")
bm25 = BM25Okapi(tokenised_corpus, k1=1.5, b=0.75)
print("Done ✓")

def bm25_retrieve(query_text: str, top_k: int = 100) -> list:
    tokens = simple_tokenize(query_text)
    scores = bm25.get_scores(tokens)          # (n_passages,)
    top_idxs = np.argsort(scores)[::-1][:top_k]
    return [idx2pid[i] for i in top_idxs]

# Sanity check
print(f"\nQuery: {sample_query}")
print(f"BM25 top-1: {pid2text.get(bm25_retrieve(sample_query)[0], 'N/A')[:200]}")

In [ ]:
bm25_scores = evaluate(bm25_retrieve, test_queries, qrels)
print("\n── BM25 Results ──")
for metric, val in sorted(bm25_scores.items()):
    print(f"  {metric}: {val:.4f}")

In [ ]:
import pandas as pd

results_df = pd.DataFrame(
    {"TF-IDF": tfidf_scores, "BM25": bm25_scores}
).T

# Sort columns nicely
col_order = [c for c in ["MRR@10", "Recall@10", "MRR@50", "Recall@50"] if c in results_df.columns]
results_df = results_df[col_order]
results_df

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 4))
results_df.plot(kind="bar", ax=ax, rot=0, colormap="tab10")
ax.set_title("Baseline Comparison — MS MARCO (test split)", fontsize=13)
ax.set_ylabel("Score")
ax.set_ylim(0, 1)
ax.legend(loc="upper right")
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig("baseline_comparison.png", dpi=150)
plt.show()
print("Saved → baseline_comparison.png")

In [ ]:
N_SHOW = 5
failures = []

for q in test_queries:
    relevant = {pid for pid, _ in qrels[q["id"]]}
    tfidf_top10 = set(tfidf_retrieve(q["text"], top_k=10))
    bm25_top10  = set(bm25_retrieve(q["text"],  top_k=10))
    # Both baselines miss
    if not tfidf_top10 & relevant and not bm25_top10 & relevant:
        failures.append(q)
    if len(failures) >= N_SHOW:
        break

print(f"Both baselines miss Recall@10 for {len(failures)} sampled queries.\n")
for q in failures:
    rel_pids = [pid for pid, _ in qrels[q["id"]]]
    print(f"Query   : {q['text']}")
    print(f"Gold pid: {rel_pids[0]}")
    print(f"Gold doc: {pid2text.get(rel_pids[0], 'N/A')[:200]}")
    print("-" * 70)